# Project Analisis Sentimen Simobi

### 1. Instalasi Library
Langkah pertama adalah menginstal library yang diperlukan untuk pemrosesan data, manipulasi array, dan pemrosesan bahasa alami (NLP).

In [16]:
!pip install pandas numpy nltk Sastrawi openpyxl -q

### 2. Import Library dan Konfigurasi NLTK
Kami mengimpor library inti seperti `pandas` dan `re`, serta mengunduh sumber daya `punkt_tab` dari NLTK yang diperlukan untuk proses tokenisasi kata.

In [17]:
import re
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize

# Download resource untuk tokenisasi kalimat/kata
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

### 3. Kamus Normalisasi (Slang Dictionary)
Untuk menangani bahasa tidak baku (slang) yang sering muncul dalam ulasan pengguna, kita mendefinisikan kamus pemetaan kata agar teks menjadi lebih standar.

In [18]:
slang_dict = {
    "tf": "transfer",
    "trf": "transfer",
    "tranfer": "transfer",
    "transfr": "transfer",
    "dftr": "daftar",
    "regis": "registrasi",
    "pftr": "daftar",
    "mutasi": "mutasi",
    "rek": "rekening",
    "norek": "rekening",
    "saldo": "saldo",
    "topup": "top-up",
    "top-up": "top-up",
    "isi-ulang": "top-up",
    "qris": "qris",
    "chris": "qris",
    "pay": "bayar",
    "biay": "biaya",
    "admn": "admin",
    "eror": "error",
    "err": "error",
    "ero": "error",
    "glitch": "error",
    "bug": "error",
    "down": "error",
    "trouble": "error",
    "gagal": "gagal",
    "ggl": "gagal",
    "pending": "tertunda",
    "pnding": "tertunda",
    "lemot": "lelet",
    "lelet": "lelet",
    "lola": "lelet",
    "loading": "lelet",
    "lama": "lama",
    "lm": "lama",
    "lme": "lama",
    "fc": "force-close",
    "kluar": "keluar",
    "keluar-sendiri": "force-close",
    "ngeblank": "blank",
    "blank": "blank",
    "muter": "loading",
    "mutar": "loading",
    " buffering": "loading",
    "blokir": "blokir",
    "keblokir": "blokir",
    "suspend": "blokir",
    "kecewa": "kecewa",
    "kcewa": "kecewa",
    "parah": "parah",
    "prh": "parah",
    "jelek": "jelek",
    "jlk": "jelek",
    "hancur": "buruk",
    "ancur": "buruk",
    "buruk": "buruk",
    "brk": "buruk",
    "payah": "payah",
    "pyh": "payah",
    "nyesel": "menyesal",
    "nyesal": "menyesal",
    "rugi": "rugi",
    "kapok": "kapok",
    "sampah": "buruk",
    "smph": "buruk",
    "ribet": "rumit",
    "rbt": "rumit",
    "susah": "sulit",
    "ssh": "sulit",
    "sulit": "sulit",
    "mantap": "bagus",
    "mntp": "bagus",
    "mantab": "bagus",
    "top": "bagus",
    "keren": "bagus",
    "krn": "bagus",
    "bagus": "bagus",
    "bgs": "bagus",
    "ok": "bagus",
    "oke": "bagus",
    "good": "bagus",
    "best": "bagus",
    "puas": "puas",
    "lancar": "lancar",
    "lncr": "lancar",
    "cepet": "cepat",
    "cept": "cepat",
    "cpt": "cepat",
    "kilat": "cepat",
    "membantu": "membantu",
    "bantu": "membantu",
    "mudah": "mudah",
    "mdh": "mudah",
    "gampang": "mudah",
    "praktis": "praktis",
    "prktis": "praktis",
    "aman": "aman",
    "yg": "yang",
    "g": "tidak",
    "ga": "tidak",
    "gak": "tidak",
    "gk": "tidak",
    "ndak": "tidak",
    "tak": "tidak",
    "tdk": "tidak",
    "bgt": "banget",
    "banget": "banget",
    "bgtu": "begitu",
    "bhg": "banget",
    "tp": "tetapi",
    "tpi": "tetapi",
    "krn": "karena",
    "karna": "karena",
    "aj": "saja",
    "aja": "saja",
    "saj": "saja",
    "utk": "untuk",
    "untk": "untuk",
    "dgn": "dengan",
    "dg": "dengan",
    "dr": "dari",
    "klo": "kalau",
    "kl": "kalau",
    "kalo": "kalau",
    "jg": "juga",
    "jga": "juga",
    "sm": "sama",
    "sma": "sama",
    "dah": "sudah",
    "udah": "sudah",
    "sdh": "sudah",
    "udh": "sudah",
    "blm": "belum",
    "belom": "belum",
    "bisa": "bisa",
    "bs": "bisa",
    "bse": "bisa",
    "skrg": "sekarang",
    "now": "sekarang",
    "pas": "saat",
    "waktu": "saat",
    "wkt": "saat",
    "aplikasiny": "aplikasinya",
    "app": "aplikasi",
    "appl": "aplikasi",
    "apk": "aplikasi",
    "aplk": "aplikasi"
}

### 4. Fungsi Pembersihan Teks (Preprocessing)
Fungsi ini melakukan beberapa tahap pembersihan:
1. Mengubah teks menjadi huruf kecil (*lowercase*).
2. Menghapus URL, mention, dan hashtag.
3. Menghapus angka dan tanda baca.
4. Melakukan tokenisasi dan normalisasi kata menggunakan kamus slang.

In [19]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()

    text = re.sub(r'http\S+|www\S+|<.*?/?>', '', text)
    text = re.sub(r'@\w+|#\w+', '', text)

    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)

    tokens = word_tokenize(text)

    normalized_tokens = [slang_dict[word] if word in slang_dict else word for word in tokens]

    cleaned_text = " ".join(normalized_tokens)

    return cleaned_text

### 5. Memuat Dataset dan Eksekusi Preprocessing
Data dimuat dari berkas CSV, baris kosong dihapus, kemudian fungsi preprocessing diterapkan pada seluruh kolom ulasan.

In [20]:
DATASET_PATH = '/content/simobi_reviews.csv'

df = pd.read_csv(DATASET_PATH)

df = df.dropna(subset=['content'])

print(f"Total data mentah yang berhasil dimuat: {len(df)} baris.")

print("Sedang memproses teks ulasan (Preprocessing)...")
df['cleaned_content'] = df['content'].apply(preprocess_text)
print("Preprocessing Selesai!")

display(df[['content', 'cleaned_content', 'score']].head())

Total data mentah yang berhasil dimuat: 10000 baris.
Sedang memproses teks ulasan (Preprocessing)...
Preprocessing Selesai!


,content,cleaned_content,score
0,mantap pokoknya,bagus pokoknya,5
1,"setelah update malah tidak bisa transfer, untu...",setelah update malah tidak bisa transfer untuk...,1
2,Parah Bank Sinarmas Tukang tipu. Saya WD dari ...,parah bank sinarmas tukang tipu saya wd dari n...,1
3,"upgrade versi terbaru gagal total, tidak bisa ...",upgrade versi terbaru gagal total tidak bisa t...,1
4,good,bagus,5


### 6. Pembagian Porsi Data Berdasarkan Rating
Pada tahap ini, kita membagi dataset menjadi tiga kategori (Negatif, Netral, Positif) untuk menjaga keseimbangan kelas. Target kita adalah mengambil total 1.000 sampel dengan distribusi sebagai berikut:
* **Negatif (Rating 1-2):** 400 sampel
* **Netral (Rating 3):** 200 sampel
* **Positif (Rating 4-5):** 400 sampel

In [21]:
df_neg = df[df['score'].isin([1, 2])]
df_neu = df[df['score'] == 3]
df_pos = df[df['score'].isin([4, 5])]

sample_neg = df_neg.sample(n=min(len(df_neg), 400), random_state=42)
sample_neu = df_neu.sample(n=min(len(df_neu), 200), random_state=42)
sample_pos = df_pos.sample(n=min(len(df_pos), 400), random_state=42)

df_manual_labeling = pd.concat([sample_neg, sample_neu, sample_pos]).sample(frac=1, random_state=42)
df_sisa_unlabeled = df[~df.index.isin(df_manual_labeling.index)].reset_index(drop=True)

print(f"✓ Berhasil memisahkan {len(df_manual_labeling)} data untuk manual labeling dan {len(df_sisa_unlabeled)} data sisa.")

✓ Berhasil memisahkan 1000 data untuk manual labeling dan 9000 data sisa.


### 7. Validasi Separasi Data
Langkah ini bertujuan untuk memastikan tidak ada tumpang tindih (*overlap*) antara data yang dipilih untuk pelabelan manual dengan data sisa yang akan digunakan untuk tahap berikutnya.

In [22]:
sisa_indeks_asli = df.index[~df.index.isin(df_manual_labeling.index)]
terjadi_duplikasi = df_manual_labeling.index.isin(sisa_indeks_asli).any()

print("=== LAPORAN DISTRIBUSI SEPARASI DATA ===")
print(f"Total Data Awal          : {len(df)} baris")
print(f"Data Sampel (Manual)     : {len(df_manual_labeling)} baris")
print(f"Sisa Data (Unlabeled)    : {len(df_sisa_unlabeled)} baris")
print(f"Apakah ada data kembar?  : {terjadi_duplikasi} (Harus False)")

=== LAPORAN DISTRIBUSI SEPARASI DATA ===
Total Data Awal          : 10000 baris
Data Sampel (Manual)     : 1000 baris
Sisa Data (Unlabeled)    : 9000 baris
Apakah ada data kembar?  : False (Harus False)


### 8. Ekspor Data ke Format CSV
Data yang telah dipisahkan kemudian disimpan ke dalam berkas CSV.
* Kolom `cleaned_content` diubah namanya menjadi `text` agar sesuai dengan format standar input Label Studio.
* `data_siap_label_manual_1000.csv` digunakan untuk pelabelan manual.
* `sisa_data_unlabeled_9000.csv` disimpan untuk proses pelabelan otomatis di masa mendatang.

In [23]:
df_manual_export = df_manual_labeling[['cleaned_content', 'score']].rename(columns={'cleaned_content': 'text'})
df_manual_export.to_csv('data_siap_label_manual_1000.csv', index=False, encoding='utf-8')

df_sisa_unlabeled[['cleaned_content', 'score']].to_csv('sisa_data_unlabeled_9000.csv', index=False, encoding='utf-8')

print("✔ Berkas CSV telah berhasil dibuat.")

✔ Berkas CSV telah berhasil dibuat.


Validasi

In [24]:
sisa_indeks_asli = df.index[~df.index.isin(df_manual_labeling.index)]
terjadi_duplikasi = df_manual_labeling.index.isin(sisa_indeks_asli).any()

print("=== LAPORAN DISTRIBUSI SEPARASI DATA ===")
print(f"Total Data Awal          : {len(df)} baris")
print(f"Data Sampel (Manual)     : {len(df_manual_labeling)} baris")
print(f"Sisa Data (Unlabeled)    : {len(df_sisa_unlabeled)} baris")
print(f"Apakah ada data kembar?  : {terjadi_duplikasi} (Harus False)")

if not terjadi_duplikasi:
    print("\n✔ Konfirmasi: Pemisahan data sudah 100% bersih!")

=== LAPORAN DISTRIBUSI SEPARASI DATA ===
Total Data Awal          : 10000 baris
Data Sampel (Manual)     : 1000 baris
Sisa Data (Unlabeled)    : 9000 baris
Apakah ada data kembar?  : False (Harus False)

✔ Konfirmasi: Pemisahan data sudah 100% bersih!
